# Fine-tuning de LLM Médica com QLoRA — Google Colab

**Objetivo:** Realizar o fine-tuning do modelo **Phi-3-mini-4k-instruct** com dados médicos sintéticos (protocolos hospitalares, FAQ clínica e laudos) utilizando a técnica **QLoRA** via biblioteca **Unsloth**, que reduz o consumo de VRAM para ~6GB — compatível com a GPU T4 gratuita do Colab.

## Fluxo do notebook
1. Instalar dependências
2. Clonar o repositório e gerar dados sintéticos
3. Pré-processar e formatar o dataset (estilo Alpaca)
4. Carregar o modelo base com quantização 4-bit
5. Aplicar adaptadores LoRA e treinar com SFTTrainer
6. Salvar o modelo e avaliar qualitativamente

> **Aviso:** Este modelo tem caráter educacional. Respostas devem ser sempre validadas por profissional de saúde.

In [ ]:
!pip install unsloth transformers datasets peft trl accelerate bitsandbytes sentencepiece -q

In [ ]:
import subprocess
subprocess.run(["git", "clone", "https://github.com/gabriel-lgomes/tech-challenge.git"])
%cd tech-challenge/fase3

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data.synthetic_generator import save_synthetic_data
from src.data.preprocessing import load_raw_dataset, build_hf_dataset, save_processed

save_synthetic_data(output_dir='data/synthetic')

raw = load_raw_dataset('data/synthetic/fine_tuning_dataset.json')
train_ds, val_ds = build_hf_dataset(raw, format='alpaca')
save_processed(train_ds, val_ds, output_dir='data/processed')

print(f'Train: {len(train_ds)} amostras | Val: {len(val_ds)} amostras')

## Fine-tuning com QLoRA (Unsloth)

Carregamos o modelo base com quantização **4-bit (NF4)** e aplicamos adaptadores **LoRA** apenas nas camadas de atenção e MLP — reduzindo os parâmetros treináveis para ~1% do total.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_from_disk

train_dataset = load_from_disk('data/processed/train')
val_dataset = load_from_disk('data/processed/val')

training_args = TrainingArguments(
    output_dir="models/fine_tuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=200,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

trainer_stats = trainer.train()
print(f"Loss final: {trainer_stats.training_loss:.4f}")

In [ ]:
model.save_pretrained("models/fine_tuned")
tokenizer.save_pretrained("models/fine_tuned")
print("Modelo salvo em models/fine_tuned/")

## Avaliação qualitativa

Testamos o modelo com perguntas clínicas reais para avaliar a qualidade das respostas após o fine-tuning.

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="models/fine_tuned", max_new_tokens=256)

perguntas = [
    "Qual é a dose de AAS para prevenção secundária cardiovascular?",
    "Quando indicar hemodiálise de urgência?",
    "Qual é o protocolo de manejo da anafilaxia?",
]

for pergunta in perguntas:
    prompt = (
        f"### Sistema:\nVocê é um assistente médico hospitalar.\n\n"
        f"### Instrução:\nResponda à pergunta clínica.\n\n"
        f"### Entrada:\n{pergunta}\n\n"
        f"### Resposta:\n"
    )
    result = pipe(prompt)[0]["generated_text"]
    resposta = result.split("### Resposta:")[-1].strip()[:400]
    print(f"\nP: {pergunta}")
    print(f"R: {resposta}")
    print("-" * 60)